Functions & Parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.mlab as mlab
from scipy.signal import butter, filtfilt, hilbert, find_peaks, coherence

from Params import get_parameters
from Eqs import run_simulation

vals = get_parameters()

(
    T, N, beta, a_e, a_i, a_th, a_rtn,
    i_e, i_i, i_th, i_rtn,
    wee, wei, wie, wii,
    weth, wthi, wthe, wrtnth, wthrtn, wertn,
    tau, dt
) = vals


cc1, cc2 = 1, 1.5

WCC = 1.65
NI = 3.9
ntrials = 10
base_seed = 42

In [ ]:
def bandpass(signal, lo, hi, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lo / nyq, hi / nyq], btype='band')
    return filtfilt(b, a, signal)


def find_threshold(signal, fs, window_sec=10.0):
    #threshold finding from quiescent window
    window_len = int(window_sec * fs)
    min_thresh = 0.24
    for trough_height in np.arange(min_thresh, 0.4, 0.01):
        w = 0
        while (w + 1) * window_len <= len(signal):
            window = signal[w * window_len:(w + 1) * window_len]
            troughs, _ = find_peaks(-window, height=trough_height)
            if len(troughs) <= 3:
                return np.mean(np.abs(window)) + 4 * np.std(np.abs(window))
            w += 1
    return -min_thresh


def detect_bursts(env, thresh, min_len, merge_gap, thresh_high): 
    #find segments where the envelope stays above the primary threshold
    segments = []
    on = env > thresh
    start = None
    for i in range(len(env)):
        if on[i]:
            if start is None:
                start = i
        elif start is not None:
            if i - start >= min_len:
                segments.append((start, i))
            start = None
    if start is not None and len(env) - start >= min_len:
        segments.append((start, len(env)))
 
    #segments with a peak above the secondary threshold
    segments2 = []
    for a, b in segments:
        peaks, _ = find_peaks(env[a:b], height=thresh_high)
        if len(peaks) >= 1:
            segments2.append((a, b))
 
    #merge segments separated by less than the merge gap
    bursts = []
    for a, b in segments2:
        if bursts and a - bursts[-1][1] <= merge_gap:
            bursts[-1] = (bursts[-1][0], b)
        else:
            bursts.append((a, b))
    return bursts


def filter_by_peaks(bursts, signal, fs, thresh, min_peaks, min_dist):
    #keep segments that have minimum number of peaks in the original signal
    peak_dist = int(round(min_dist * fs))
    kept = []
    for a, b in bursts:
        peaks, _ = find_peaks(np.abs(signal[a:b]), height=thresh, distance=peak_dist)
        if len(peaks) >= min_peaks:
            kept.append((a, b))
    return kept


def detect_seizures(signal, fs, raw_thresh):
    #full burst-detection steps
    filt = bandpass(signal, 5.0, 10.0, fs)
    env = np.abs(hilbert(filt))

    mean_env, std_env = np.mean(env), np.std(env)
    thresh = mean_env + 0.1 * std_env
    thresh_high = mean_env + 1.5 * std_env
    min_len = int(1 * (1.0 / 7.5) * fs)
    merge_gap = int(0.2 * fs)

    bursts = detect_bursts(env, thresh, min_len, merge_gap, thresh_high)
    bursts = filter_by_peaks(bursts, signal, fs, raw_thresh, 4, 0.1)
    return env, filt, bursts, thresh, thresh_high


def interictal_segments(bursts, n):
    gaps = []
    last = 0
    for a, b in bursts:
        if a > last:
            gaps.append((last, a))
        last = b
    if last < n:
        gaps.append((last, n))
    return gaps


def mean_coherence(segments, e0, e1, fs, nperseg, edges, Kmin=4):
    #mean coherence across segments
    nperseg = int(nperseg)
    noverlap = nperseg // 2
    step = nperseg - noverlap
    coh_list = []
    f_ref = None
    for a, b in segments:
        n_windows = 1 + (b - a - nperseg) // step
        if n_windows < Kmin:
            continue
        f, Cxy = coherence(e0[a:b], e1[a:b], fs=fs, nperseg=nperseg, noverlap=noverlap, nfft=nperseg)
        if f_ref is None:
            f_ref = f
        coh_list.append(np.nan_to_num(Cxy))
    n_bins = len(edges) - 1
    if not coh_list:
        return np.full(n_bins, np.nan)
    mean_coh = np.nanmean(np.vstack(coh_list), axis=0)
    out = np.full(n_bins, np.nan)
    for k, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (f_ref >= lo) & (f_ref < hi)
        if np.any(mask):
            out[k] = np.mean(mean_coh[mask])
    return out

# **Figure 3f,g, Supplemental a,b**

In [ ]:
ictal = {cc1: [], cc2: []}
inter = {cc1: [], cc2: []}
plot_data = {}

for trial in range(ntrials):
    for cv in (cc1, cc2):
        seed = base_seed + 2 * trial + (0 if cv == cc1 else 1)

        signals = run_simulation(
            num_delays=100, cv=cv,
            T=T, N=N, wcc=WCC, beta=beta,
            a_e=a_e, a_i=a_i, a_th=a_th, a_rtn=a_rtn,
            i_e=i_e, i_i=i_i, i_th=i_th, i_rtn=i_rtn,
            wee=wee, wei=wei, wie=wie, wii=wii,
            weth=weth, wthi=wthi, wthe=wthe,
            wrtnth=wrtnth, wthrtn=wthrtn, wertn=wertn,
            tau=tau, noise_intensity=NI, dt=dt, seed=seed,
        )

        #downsample to 1000 Hz and drop the transient
        start = int(round(24000))
        e0 = signals[0, start::4].copy()
        e1 = signals[1, start::4].copy()
        fs = 1000.0
        t = np.arange(len(e0)) / fs

        thr = find_threshold(e0, fs)
        env, filt, bursts, th1, th2 = detect_seizures(e0, fs, thr)

        bin_edges = np.arange(1, 81 + 2, 2)

        ci = mean_coherence(bursts, e0, e1, fs, int(1.024 * fs), bin_edges)
        interictal = interictal_segments(bursts, len(e0))
        cj = mean_coherence(interictal, e0, e1, fs, int(1.024 * fs) * 4, bin_edges)
        if not np.all(np.isnan(ci)):
            ictal[cv].append(ci)
        if not np.all(np.isnan(cj)):
            inter[cv].append(cj)

        if trial == 0:
            #spectrogram, z-scored per frequency
            NFFT = int(round(1.024 * fs))
            Pxx, freqs, times = mlab.specgram(e0, NFFT=NFFT, Fs=fs, noverlap=int(0.95 * NFFT))
            Sxx = 10 * np.log10(Pxx + 1e-20)
            Sxx = (Sxx - Sxx.mean(1, keepdims=True)) / Sxx.std(1, keepdims=True)
            plot_data[cv] = dict(e0=e0, t=t, fs=fs, env=env, filt=filt, bursts=bursts,
                          th1=th1, th2=th2, thr=thr, freqs=freqs, times=times, Sxx=Sxx)

x1, x2 = 710, 1020
for cv in (cc1, cc2):
    cv_data = plot_data[cv]
    plt.figure(figsize=(12, 9))

    plt.subplot(2, 1, 1)
    idx = np.where((cv_data["times"] >= x1) & (cv_data["times"] <= x2))[0]
    lo, hi = max(0, idx[0] - 1), min(len(cv_data["times"]), idx[-1] + 2)
    plt.pcolormesh(cv_data["times"][lo:hi], cv_data["freqs"], cv_data["Sxx"][:, lo:hi],
                   cmap='hot', vmin=1, vmax=3, shading='gouraud')
    plt.ylim(0, 50)
    plt.xlim(x1, x2)
    plt.ylabel("Frequency (Hz)")
    plt.gca().spines[['top', 'right']].set_visible(False)

    plt.subplot(2, 1, 2)
    plt.plot(cv_data["t"], cv_data["e0"], label='raw')
    plt.plot(cv_data["t"], cv_data["filt"], alpha=0.7, label='filtered')
    plt.plot(cv_data["t"], cv_data["env"], color='red', alpha=0.7, label='envelope')
    plt.axhline(cv_data["th1"], color='green', ls='--')
    plt.axhline(cv_data["th2"], color='purple', ls='--')
    plt.axhline(-cv_data["thr"], color='black', ls='--')
    for s, e in cv_data["bursts"]:
        plt.axvspan(cv_data["t"][s], cv_data["t"][e], color='yellow', alpha=0.3)
    plt.xlim(x1, x2)
    plt.ylim(-0.8, 0.25)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.gca().spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()


#Histogram burst durations and intervals
col = {cc1: "tab:blue", cc2: "tab:red"}
alpha = {cc1: 1.0, cc2: 0.7}
plt.figure(figsize=(12, 5))
for cv in (cc1, cc2):
    cv_data = plot_data[cv]
    burst_times = [(cv_data["t"][s], cv_data["t"][e]) for s, e in cv_data["bursts"]]
    dur = [e - s for s, e in burst_times]
    intervals = [burst_times[i][0] - burst_times[i - 1][1] for i in range(1, len(burst_times))]
    plt.subplot(1, 2, 1)
    plt.hist(dur, bins=50, range=(0, 25), alpha=alpha[cv], color=col[cv], label=f"CV = {cv} m/s")
    plt.xlim(0, 10)
    plt.ylim(0, 100)
    plt.subplot(1, 2, 2)
    plt.hist(intervals, bins=50, range=(0, 100), alpha=alpha[cv], color=col[cv], label=f"CV = {cv} m/s")
    plt.xlim(0, 100)
    plt.ylim(0, 50)
plt.subplot(1, 2, 1)
plt.xlim(0, 10); plt.xlabel("Burst duration (s)"); plt.ylabel("Count"); plt.legend()
plt.gca().spines[['top', 'right']].set_visible(False)
plt.subplot(1, 2, 2)
plt.xlim(0, 100); plt.xlabel("Inter-burst interval (s)"); plt.ylabel("Count"); plt.legend()
plt.gca().spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()


#mean coherence across trials
fig, ax = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for a, data, title in ((ax[0], ictal, "ictal"), (ax[1], inter, "interictal")):
    for cv in (cc1, cc2):
        if not data[cv]:
            continue
        arr = np.stack(data[cv])
        m = np.nanmean(arr, 0)
        sem = np.nanstd(arr, 0) / np.sqrt(arr.shape[0])
        bin_centers = np.arange(2, 81, 2)
        a.plot(bin_centers, m, lw=2, color=col[cv], label=f"CV = {cv} m/s")
        a.fill_between(bin_centers, m - sem, m + sem, color=col[cv], alpha=0.25)
    a.set_title(f"{title} coherence"); a.set_xlabel("Frequency (Hz)")
    a.set_xlim(1, 30); a.set_ylim(0, 1); a.legend()
    a.spines[['top', 'right']].set_visible(False)
ax[0].set_ylabel("Coherence")
plt.tight_layout()
plt.show()

# **Figure 3h,i**

In [ ]:
cv_values = np.round(np.arange(1.0, 1.501, 0.1), 2)

rate_mean, rate_sem = [], []
dur_mean, dur_sem = [], []

for cv_idx, cv in enumerate(cv_values):
    rates, durs = [], []
    for trial in range(ntrials):
        seed = base_seed + trial * len(cv_values) + cv_idx

        signals = run_simulation(
            num_delays=100, cv=cv,
            T=T, N=N, wcc=WCC, beta=beta,
            a_e=a_e, a_i=a_i, a_th=a_th, a_rtn=a_rtn,
            i_e=i_e, i_i=i_i, i_th=i_th, i_rtn=i_rtn,
            wee=wee, wei=wei, wie=wie, wii=wii,
            weth=weth, wthi=wthi, wthe=wthe,
            wrtnth=wrtnth, wthrtn=wthrtn, wertn=wertn,
            tau=tau, noise_intensity=NI, dt=dt, seed=seed,
        )

        e0 = signals[0, 24000::4].copy()
        fs = 1000.0

        thr = find_threshold(e0, fs)
        _, _, bursts, _, _ = detect_seizures(e0, fs, thr)

        hours = (len(e0) / fs) / 3600.0
        rates.append(len(bursts) / hours)
        if bursts:
            durs.append(np.mean([(e - s) / fs for s, e in bursts]))
        else:
            durs.append(np.nan)

    rates = np.array(rates)
    durs = np.array(durs)
    n_dur = np.sum(~np.isnan(durs))

    rate_mean.append(rates.mean())
    rate_sem.append(rates.std() / np.sqrt(len(rates)))
    dur_mean.append(np.nanmean(durs))
    dur_sem.append(np.nanstd(durs) / np.sqrt(n_dur))

rate_mean = np.array(rate_mean); rate_sem = np.array(rate_sem)
dur_mean = np.array(dur_mean); dur_sem = np.array(dur_sem)

for cv, r, d in zip(cv_values, rate_mean, dur_mean):
    print(f"CV={cv}: rate={r:.2f}/h  duration={d:.3f}s")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.errorbar(cv_values, rate_mean, yerr=rate_sem, fmt='-o', capsize=5, color='blue')
ax1.set_xlabel('CV (m/s)'); ax1.set_ylabel('Seizures per hour'); ax1.set_title('Seizure rate')
ax1.spines[['top', 'right']].set_visible(False)
ax2.errorbar(cv_values, dur_mean, yerr=dur_sem, fmt='-o', capsize=5, color='red')
ax2.set_xlabel('CV (m/s)'); ax2.set_ylabel('Duration (s)'); ax2.set_title('Seizure duration')
ax2.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); 
plt.show()